# 03 — Your First MCP Client Connection

Now we write the smallest possible Python MCP *client*: it launches the Azure MCP Server as a subprocess over stdio and completes the `initialize()` handshake from notebook 01, step 1.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.mcp_client import AzureMcpClient, build_server_params

In [ ]:
# build_server_params() just builds the `npx ... server start` command line -
# no network call happens yet.
params = build_server_params(read_only=True)
print(params.command, params.args)

In [ ]:
# This performs the real connection: it spawns the server process, opens a
# stdio JSON-RPC session, and calls initialize(). Requires Node.js + `az login`.
async def connect_once():
    async with AzureMcpClient(params) as client:
        print("Connected! Session initialized:", client.session is not None)

try:
    await connect_once()
except Exception as exc:
    print(f"Could not connect (expected if Node.js/az login aren't set up yet): {exc}")

## What just happened?

`AzureMcpClient.__aenter__` did three things: spawned `npx @azure/mcp@latest server start` as a child process, wrapped its stdin/stdout in an MCP `ClientSession`, and called `session.initialize()` to negotiate the protocol version. When the `async with` block exits, both the session and the child process are cleanly torn down.

## Troubleshooting & next steps

- **`FileNotFoundError` for `npx`** — install Node.js LTS.
- **Hangs on connect** — the very first `npx @azure/mcp@latest` run downloads the package; be patient or pre-warm it from a terminal.
- **Authorization errors later on** — revisit [`docs/04_authentication.md`](../docs/04_authentication.md).

Continue to [`04_exploring_available_tools.ipynb`](04_exploring_available_tools.ipynb).